In [ ]:


import numpy as np


import matplotlib.pyplot as plt


import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
from config import *
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in Path(folder_loc).parents if (p / 'jar_locations.py').is_file())))
from generation import generate_arguments_string, run_simulation
pop_sizes_loc = next(p for p in Path(folder_loc).parents if (p / 'jar_locations.py').is_file()) / 'pop_sizes' / 'weak_selection'
import results_access

from jar_locations import simEMoranFix_jar


In [ ]:
def get_fix_probs_and_times(graph_code, pop_size, deathrate, r, overwrite=False, samples=samples, resident=1):
    if overwrite:
        raise ValueError("Existing results are not overwritten; move them aside explicitly first.")
    code = graph_code.replace('F', 'upper_F').replace('f', 'lower_f').replace('M', 'upper_M')
    output_name = f"eMoran-N-{pop_size}-geom-{code}-d-{deathrate}-r-{r}-resident-{resident}-tr-{timerelax}"
    result = Path(results_loc+output_name+'.txt')
    if result.exists():
        count = results_access.get_sample_size_weighted(output_name, results_loc)
        if count < samples:
            raise ValueError(f"{result} contains {count} samples; {samples} are required.")
        return
    arguments_dict = {
        'timerelax': timerelax,
        'popsize': pop_size,
        'deathrate': deathrate,
        'module': 'eMoran',
        'digits': 6,
        'delay': 0,
        'timestep': 1,
        'popupdate': 'e',
        'geometry': graph_code,
        'fitness': f'{resident},{r}',
        'mutation': 0,
        'init': ('mutant', '1,0'),
        'monostop': None,
        'samples': samples
    }
    run_simulation(results_loc, output_name, simEMoranFix_jar, generate_arguments_string(arguments_dict))

def get_M_directly(graph,d,k,resident=1,regular=True):
    print("Getting M directly for graph", graph)
    if regular:
        graph_code = f"r{k}"
        pop_size = int(np.round(K/(1-(k-1)*d/(k-1-d))))
    elif k is not None:
        graph_code = f"{graph}"
        pop_size = int(np.round(K/(1-(k-1)*d/(k-1-d))))
    else:
        graph_code =graph
        pop_size = K
    current_expected_val = 1
    while abs(current_expected_val-K)>1:
        get_fix_probs_and_times(graph_code,pop_size,d,0,overwrite=False,samples=samples,resident=resident)
        output_name = f"eMoran-N-{pop_size}-geom-{graph_code.replace('F', 'upper_F').replace('f', 'lower_f').replace("M", "upper_M")}-d-{d}-r-{0}-resident-{resident}-tr-{timerelax}"
        with open(f'{results_loc}{output_name}.txt', 'r') as file:
            lines = file.readlines()
        total_samples = 0
        total_samples_and_sizes = 0
        for line in lines:
            split_line = line.split(',')
            try:
                key = float(split_line[0].strip())
                val = float(split_line[1].strip())+float(split_line[2].strip())
                total_samples+=val
                total_samples_and_sizes+=key*val
            except Exception as e:
                # print(e)
                pass
        if total_samples_and_sizes ==0:
            raise ValueError("No valid data found in file.")
        current_expected_val = total_samples_and_sizes / total_samples
        if abs(current_expected_val-K)>1:
            print('Current K', current_expected_val, 'Current M', pop_size)
            pop_size = int(np.round(pop_size*K/current_expected_val))
    
    return pop_size

In [ ]:
save_fig = True
save_fig_name = "lattice_both"
overwriting = False

# --- Matplotlib settings for publication ---
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "lines.linewidth": 1.5,
    "figure.figsize": (6.25, 5.25), 
    "figure.dpi": 300,
})

rs = np.array([0.95,0.96,0.97,0.98,0.99,1.0,1.01,1.02,1.03,1.04,1.05])
deathrates = [0.2,0.5]

linestyles = ['-','--', '-.', ':', '--','--']
dashlist = [None,None, None, None, (4,7), (5,2,20,2)]
orig_cols = ['#a38fc4', '#6883ba', '#3a7aa8', '#1e6285', '#033f63']
cols = ['black',orig_cols[1],orig_cols[3],orig_cols[1],orig_cols[3],orig_cols[1]]

for d_ind, d in enumerate(deathrates):
    fig2, ax2 = plt.subplots(figsize=(5, 3.5))
    k_slopes = []
    ks_for_slopes = []
    f_slopes = []
    fs_for_slopes = []
    for g_ind, graph in enumerate(graphs):
        graph_Ks = []
        graph_neutral = None
        fname = pop_sizes_loc / f'pop_size_{graph["code"].replace("F","upper_F").replace("M","upper_M")}_d_{d}.txt'
        try:
            with open(fname, 'r') as f:
                pop_size = int(f.read())
        except FileNotFoundError:
            if graph["code"][0] == "r":
                pop_size = get_M_directly(graph['code'], d, int(graph["code"][1:]), regular=True)
            elif "equivalent_k" in graph:
                pop_size = get_M_directly(graph['code'], d, graph["equivalent_k"], regular=False)
            else:
                pop_size = get_M_directly(graph['code'], d, 100, regular=False)

            with open(fname, 'w') as f:
                f.write(str(pop_size))

        for r in rs:
            get_fix_probs_and_times(
                graph['code'], pop_size, d, 0,
                overwrite=overwriting, samples=5000, resident=r
            )

            output_name = (
                f"eMoran-N-{pop_size}-geom-{graph['code'].replace('F','upper_F').replace('f','lower_f').replace('M','upper_M')}"
                f"-d-{d}-r-0-resident-{r}-tr-{timerelax}"
            )

            with open(f'{results_loc}{output_name}.txt', 'r') as file:
                lines = file.readlines()

            total_samples = 0
            total_samples_and_sizes = 0

            for line in lines:
                split_line = line.split(',')
                try:
                    key = float(split_line[0].strip())
                    val = float(split_line[1].strip()) + float(split_line[2].strip())
                    total_samples += val
                    total_samples_and_sizes += key * val
                except:
                    pass

            if total_samples_and_sizes == 0:
                raise ValueError("No valid data found in file.")

            current_expected_val = total_samples_and_sizes / total_samples
            graph_Ks.append(current_expected_val)

            if r == 1:
                graph_neutral = current_expected_val

        normalized = np.array(graph_Ks) / graph_neutral
        linestyle = linestyles[g_ind % len(linestyles)]
        color = cols[g_ind %len(cols)]
        dashstyle = dashlist[g_ind % len(dashlist)]

        slope = np.polyfit(rs, normalized, 1)[0]
        print(f"Slope for graph {graph['name']} (code: {graph['code']}) at death rate {d}: {slope}")
        if graph['code'] == 'M':
            ax2.axhline(slope, color='black', linestyle='--', alpha=0.5, label=rf"Well-mixed $\alpha={slope:.4f}$")
        elif graph['code'][0] == 'r':
            ks_for_slopes.append(int(graph['code'][1:]))
            k_slopes.append(slope)
        elif graph['code'] == 'n':
            ax2.scatter([4], [slope], color='black', marker="+", label=rf"{graph['name']}", alpha=0.8)
        elif graph['code'] == 'm':
            ax2.scatter([8], [slope], color='black', marker="o", label=rf"{graph['name']}", alpha=0.8,facecolors='none')
        elif graph['code'][0] == 'f':
            fs_for_slopes.append(int(graph['code'][1:]))
            f_slopes.append(slope)
    
    ax2.scatter(ks_for_slopes, k_slopes, color='black', marker='x', label=r"Regular Graphs", alpha=0.8)
    krange = np.linspace(min(ks_for_slopes)-1, max(ks_for_slopes)+1, 100)
    ax2.plot(krange,d/((1-d/(krange-1))*(1-d*krange/(krange-1))), color='gray', linestyle=':', linewidth=2, alpha=0.7, label=r"Pair Approx. $\alpha = \frac{d}{{(1-\frac{d}{(k-1)})(1-\frac{d k}{(k-1)})}}$")
    ax2.set_xlabel(r"Degree $k$")
    ax2.set_ylabel(r"Equilibrium Density ratio slope $\alpha$")
    ax2.legend(frameon=False,fontsize="medium")
    fig2.tight_layout()
    fig2.savefig(f"alpha_vs_k_d_{d}_{save_fig_name}.pdf", bbox_inches='tight')


